In [ ]:
import subprocess
import threading
import re
import time

# Start Cloudflare Quick Tunnel
tunnel_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8000",
        "--no-autoupdate"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

tunnel_lines = []

def read_tunnel_output():
    for line in tunnel_process.stdout:
        tunnel_lines.append(line)

threading.Thread(
    target=read_tunnel_output,
    daemon=True
).start()

public_url = None

for _ in range(30):
    output = "\n".join(tunnel_lines)

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        output
    )

    if match:
        public_url = match.group(0)
        break

    if tunnel_process.poll() is not None:
        break

    time.sleep(1)

if public_url:
    print("Your public API URL:")
    print(public_url)
else:
    print("Tunnel URL not found. Recent output:")
    print("\n".join(tunnel_lines[-15:]))

In [13]:
if public_url:
    response = requests.get(
        public_url + "/students/101",
        timeout=30
    )

    print("Status:", response.status_code)
    print(response.json())

Status: 200
{'name': 'Raghava', 'age': 20, 'grade': 'A', 'department': 'CSE'}


In [14]:
tunnel_process.terminate()

try:
    tunnel_process.wait(timeout=10)
except subprocess.TimeoutExpired:
    tunnel_process.kill()

print("Public tunnel stopped.")

Public tunnel stopped.


In [16]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

from pydantic import BaseModel, Field

app = FastAPI(
    title="AI-Powered University API",
    description=(
        "Student management, machine learning, "
        "chatbot and RAG APIs"
    ),
    version="1.0.0"
)


# -----------------------------------------
# DATA MODELS
# -----------------------------------------

class StudentCreate(BaseModel):

    name: str = Field(min_length=2)
    age: int = Field(ge=16, le=100)
    department: str
    cgpa: float = Field(ge=0, le=10)


class StudentPatch(BaseModel):

    name: str | None = Field(default=None, min_length=2)
    age: int | None = Field(default=None, ge=16, le=100)
    department: str | None = None
    cgpa: float | None = Field(default=None, ge=0, le=10)


class IrisInput(BaseModel):

    sepal_length: float = Field(gt=0)
    sepal_width: float = Field(gt=0)
    petal_length: float = Field(gt=0)
    petal_width: float = Field(gt=0)


class ChatRequest(BaseModel):

    question: str = Field(min_length=1, max_length=1000)


class RAGRequest(BaseModel):

    question: str = Field(min_length=1, max_length=1000)


# -----------------------------------------
# IN-MEMORY DATABASE
# -----------------------------------------

students_db = {}

next_student_id = 101


# -----------------------------------------
# BASIC ENDPOINTS
# -----------------------------------------

@app.get("/")
def home():

    return {
        "message": "AI University API",
        "version": "1.0.0"
    }


@app.get("/health")
def health():

    return {
        "status": "ok"
    }


# -----------------------------------------
# STUDENT MANAGEMENT
# -----------------------------------------

@app.get("/students")
def list_students(department: str | None = None):

    results = []

    for student_id, student in students_db.items():

        if department is not None:

            if student["department"] != department:
                continue

        results.append({
            "student_id": student_id,
            **student
        })

    return results


@app.get("/students/{student_id}")
def get_student(student_id: int):

    if student_id not in students_db:

        raise HTTPException(
            status_code=404,
            detail="Student not found"
        )

    return {
        "student_id": student_id,
        **students_db[student_id]
    }


@app.post("/students", status_code=201)
def create_student(student: StudentCreate):

    global next_student_id

    student_id = next_student_id

    next_student_id += 1

    students_db[student_id] = student.model_dump()

    return {
        "student_id": student_id,
        **students_db[student_id]
    }


@app.put("/students/{student_id}")
def replace_student(
    student_id: int,
    student: StudentCreate
):

    if student_id not in students_db:

        raise HTTPException(
            status_code=404,
            detail="Student not found"
        )

    students_db[student_id] = student.model_dump()

    return {
        "student_id": student_id,
        **students_db[student_id]
    }


@app.patch("/students/{student_id}")
def patch_student(
    student_id: int,
    update: StudentPatch
):

    if student_id not in students_db:

        raise HTTPException(
            status_code=404,
            detail="Student not found"
        )

    changes = update.model_dump(exclude_unset=True)

    candidate = {
        **students_db[student_id],
        **changes
    }

    validated = StudentCreate.model_validate(candidate)

    students_db[student_id] = validated.model_dump()

    return {
        "student_id": student_id,
        **students_db[student_id]
    }


@app.delete("/students/{student_id}")
def delete_student(student_id: int):

    if student_id not in students_db:

        raise HTTPException(
            status_code=404,
            detail="Student not found"
        )

    del students_db[student_id]

    return {
        "message": "Student deleted"
    }


# -----------------------------------------
# MACHINE LEARNING API
# -----------------------------------------

@app.post("/predict-iris")
def predict_iris(data: IrisInput):

    features = [[
        data.sepal_length,
        data.sepal_width,
        data.petal_length,
        data.petal_width
    ]]

    prediction = classifier.predict(features)[0]

    probabilities = classifier.predict_proba(features)[0]

    return {
        "predicted_class": iris.target_names[prediction],
        "probabilities": {
            name: float(prob)
            for name, prob in zip(
                iris.target_names,
                probabilities
            )
        }
    }


# -----------------------------------------
# CHATBOT API
# -----------------------------------------

@app.post("/chat")
def chat(request: ChatRequest):

    answer = generate_answer(
        request.question
    )

    return {
        "question": request.question,
        "answer": answer
    }


# -----------------------------------------
# RAG API
# -----------------------------------------

@app.post("/rag")
def rag(request: RAGRequest):

    context = retrieve_context(
        request.question
    )

    prompt = f"""
Use only the following context.

Context:
{chr(10).join(context)}

Question:
{request.question}

If the answer is unavailable, say so.
"""

    answer = generate_answer(prompt)

    return {
        "question": request.question,
        "answer": answer,
        "context": context
    }


# -----------------------------------------
# TEST CLIENT
# -----------------------------------------

client = TestClient(app)

print("Application created successfully!")

print("Available endpoints:")

for route in app.routes:

    if hasattr(route, "methods"):

        print(
            route.path,
            route.methods
        )

Application created successfully!
Available endpoints:
/openapi.json {'HEAD', 'GET'}
/docs {'HEAD', 'GET'}
/docs/oauth2-redirect {'HEAD', 'GET'}
/redoc {'HEAD', 'GET'}
/ {'GET'}
/health {'GET'}
/students {'GET'}
/students/{student_id} {'GET'}
/students {'POST'}
/students/{student_id} {'PUT'}
/students/{student_id} {'PATCH'}
/students/{student_id} {'DELETE'}
/predict-iris {'POST'}
/chat {'POST'}
/rag {'POST'}


/Users/mukundanramesh/projects/ai_ml_course_bootcamp_practice/aiml_venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [17]:
import threading
import time
import uvicorn
import requests

PORT = 8000

def run_server():
    uvicorn.run(
        app,
        host="127.0.0.1",
        port=PORT,
        log_level="warning"
    )

server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

time.sleep(3)

response = requests.get(
    f"http://127.0.0.1:{PORT}/",
    timeout=10
)

print(response.json())

ERROR:    [Errno 48] error while attempting to bind on address ('127.0.0.1', 8000): address already in use


{'message': 'Student API is running'}


In [18]:
import subprocess
import threading
import re
import time

# Start Cloudflare Quick Tunnel
tunnel_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8000",
        "--no-autoupdate"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

tunnel_lines = []

def read_tunnel_output():
    for line in tunnel_process.stdout:
        tunnel_lines.append(line)

threading.Thread(
    target=read_tunnel_output,
    daemon=True
).start()

public_url = None

for _ in range(30):
    output = "\n".join(tunnel_lines)

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        output
    )

    if match:
        public_url = match.group(0)
        break

    if tunnel_process.poll() is not None:
        break

    time.sleep(1)

if public_url:
    print("Your public API URL:")
    print(public_url)
else:
    print("Tunnel URL not found. Recent output:")
    print("\n".join(tunnel_lines[-15:]))

FileNotFoundError: [Errno 2] No such file or directory: 'cloudflared'